# 🔥 Vak AI - Fine-Tuning Qwen 2.5 (3B) as Chill Guy / Gen-Z with Real Emojis
### ⚡ Hardware Target: Kaggle Dual T4 GPUs (2x 16GB VRAM = 32GB VRAM)
### 🛠️ Method: QLoRA (4-bit NF4) + LoRA (r=16, alpha=32) + ChatML Completion Loss Masking

This notebook turns **Qwen2.5-3B-Instruct** into **Vak**: an unbothered, funny, high-IQ AI companion who speaks fluent Gen-Z slang and uses authentic emojis (💀, 😭, 🗿, 🫡, 💅, 🔥, 😮‍💨, 🧢, 🤝) while keeping 100% of its coding, math, and reasoning intelligence intact.

In [ ]:
# 1. Install required packages
!pip install -q -U torch transformers peft trl bitsandbytes accelerate datasets huggingface_hub


In [ ]:
# 2. Check GPU Environment (Dual T4s)
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# 3. Load Dataset
import os
from datasets import load_dataset

# Specified Kaggle dataset input path with resilient fallback
dataset_file = "/kaggle/input/datasets/bhuvanagiribharadwaj/genz-dataset-jsonl/genz_dataset.jsonl"
if not os.path.exists(dataset_file):
    alternatives = [
        "genz_dataset.jsonl",
        "/kaggle/input/genz-dataset-jsonl/genz_dataset.jsonl",
        "data/genz_dataset.jsonl"
    ]
    for alt in alternatives:
        if os.path.exists(alt):
            dataset_file = alt
            break

print(f"📂 Loading dataset from: {dataset_file}")
dataset = load_dataset("json", data_files=dataset_file, split="train")
split_data = dataset.train_test_split(test_size=0.05, seed=42)
print(f"✅ Training samples: {len(split_data['train'])}, Validation samples: {len(split_data['test'])}")


In [ ]:
# 4. Setup Model, Tokenizer & 4-bit QLoRA
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "vak_qwen3b_genz_adapter"

# 4-bit NormalFloat quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, padding_side="right")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load Model sharded automatically across both T4 GPUs
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

# LoRA Config targeting all linear projection layers
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# 5. Format ChatML & Response-Only Loss Masking
import torch
from transformers import DataCollatorForLanguageModeling

# Robust, drop-in completion-only collator (compatible with all modern TRL & Transformers releases)
class DataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):
    def __init__(self, response_template, tokenizer, mlm=False, ignore_index=-100):
        super().__init__(tokenizer=tokenizer, mlm=mlm)
        self.response_template = response_template
        if isinstance(response_template, str):
            self.response_token_ids = tokenizer.encode(response_template, add_special_tokens=False)
        else:
            self.response_token_ids = response_template
        self.ignore_index = ignore_index

    def torch_call(self, examples):
        batch = super().torch_call(examples)
        response_len = len(self.response_token_ids)
        for i in range(len(examples)):
            labels = batch["labels"][i]
            matched = False
            start_indices = (labels == self.response_token_ids[0]).nonzero(as_tuple=True)[0]
            for idx in start_indices:
                idx = idx.item()
                if labels[idx : idx + response_len].tolist() == self.response_token_ids:
                    labels[: idx + response_len] = self.ignore_index
                    matched = True
                    break
            if not matched:
                labels[:] = self.ignore_index
        return batch

def format_chatml(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_data = split_data["train"].map(format_chatml)
eval_data = split_data["test"].map(format_chatml)

# Safeguard: train strictly on assistant completion tokens, never on user prompt tokens!
response_template = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(response_template=response_template, tokenizer=tokenizer)


In [ ]:
# 6. Training Execution (Dual GPU Optimized)
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=1024,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    num_train_epochs=2,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit"
)

# Note: peft_config is omitted here because model was already wrapped with get_peft_model() in Cell 4
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    processing_class=tokenizer,
    data_collator=collator,
    args=training_args
)

print("🚀 Starting QLoRA Fine-Tuning...")
trainer.train()


In [ ]:
# 7. Save Adapter Locally, Create Zip & Push to Hugging Face Hub
import os
from huggingface_hub import login, HfApi

HF_REPO_ID = "blaze1234567/qwen_chill_guy"
HF_TOKEN = "YOUR_HF_TOKEN"

# 1. Save adapter weights and tokenizer locally
print(f"💾 Saving adapter weights locally to {OUTPUT_DIR}...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# 2. Authenticate and push PEFT adapter to Hugging Face Hub
print(f"🚀 Authenticating and pushing adapter to Hugging Face Hub: {HF_REPO_ID}...")
login(token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)

trainer.model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)

# 3. Zip archive backup for direct Kaggle download
!zip -r vak_qwen3b_genz_adapter.zip {OUTPUT_DIR}
print(f"\n🎉 SUCCESS! Adapter published to: https://huggingface.co/{HF_REPO_ID}")
print(f"📦 Local zip backup ready: vak_qwen3b_genz_adapter.zip")


In [ ]:
# 8. Live Interactive Verification (Check Persona, Emojis & Zero-Overfitting)
from transformers import pipeline

test_prompts = [
    "yo what's good?",
    "Can you explain quantum computing simply?",
    "How do I reverse a string in Python?",
    "Bro I am so stressed for my interview tomorrow 😭"
]

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
system_msg = "You are Vak, a super chill, unbothered Gen-Z AI assistant. You keep things 100% real, speak with effortless slang and emojis (💀, 😭, 🗿, 🫡, 💅, 🔥, 😮‍💨), but always give accurate high-IQ answers."

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": prompt}
    ]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    output = pipe(formatted, max_new_tokens=256, temperature=0.7, top_p=0.9, do_sample=True)
    
    response = output[0]["generated_text"][len(formatted):]
    print(f"\n🧑 USER: {prompt}")
    print(f"🤖 VAK: {response.strip()}")
    print("=" * 60)